# Service Risk Monitoring

## Objective

Monitor customer service performance and identify fulfillment risks that may impact customer satisfaction.

This module evaluates the ability of the supply chain network to fulfill customer demand based on inventory availability and capacity conditions.

## Key Questions

- Which products are at risk of stockout?
- Which plants are unable to support customer demand?
- Which products may experience fulfillment issues?
- What are the major drivers of service risk?

## Data Sources

- inventory_risk_weekly.csv
- capacity_utilization_weekly.csv
- top_product_driver_weekly.csv

## Deliverables

- service_risk_weekly.csv
- service_alerts.csv
- dashboard_service_summary.csv

In [1]:
import pandas as pd
import numpy as np

In [2]:
inventory_risk = pd.read_csv(
    "../data/processed/inventory_risk_weekly.csv"
)

capacity_utilization = pd.read_csv(
    "../data/processed/capacity_utilization_weekly.csv"
)

product_plant_mapping = pd.read_csv(
    "../data/master/product_plant_mapping.csv"
)

In [5]:
service_risk = (
    inventory_risk.merge(
        capacity_utilization[
            [
                "Plant_ID",
                "Order_Date",
                "Capacity_Status"
            ]
        ],
        on=[
            "Plant_ID",
            "Order_Date"
        ],
        how="left"
    )
)

service_risk.head()

,Plant_ID,Product Card Id,Product Name,Order_Date,Weekly_Demand,Initial_Inventory,Target_Coverage_Days,Average_Daily_Demand,Review_Cycle,Beginning_Inventory,...,Target_Inventory,Safety_Stock_Days,Inventory_On_Hand,DOI,Inventory_Status,Reorder_Risk,ABC_Class,Inventory_Utilization,Inventory_Alert,Capacity_Status
0,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-04-30,2,48,21,0.457143,0,48.0,...,10,7,46.0,100.6250,Normal,No,C,4.6,0,Normal
1,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-07,3,48,21,0.457143,0,46.0,...,10,7,43.0,94.0625,Normal,No,C,4.3,0,Normal
2,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-14,1,48,21,0.457143,0,43.0,...,10,7,42.0,91.8750,Normal,No,C,4.2,0,Normal
3,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-21,3,48,21,0.457143,0,42.0,...,10,7,39.0,85.3125,Normal,No,C,3.9,0,Normal
4,P2,19,Nike Men's Fingertrap Max Training Shoe,2017-05-28,6,48,21,0.457143,1,39.0,...,10,7,33.0,72.1875,Normal,No,C,3.3,0,Normal


## Service Risk Classification

### High

Inventory_Status = Critical

AND

Capacity_Status = Critical

---

### Medium

Inventory_Status = Critical

OR

Capacity_Status = Critical

---

### Low

No critical inventory or capacity alerts.

In [6]:
# Overall fulfillment risk level

def classify_service_risk(row):

    if (
        row["Inventory_Status"] == "Critical"
        and
        row["Capacity_Status"] == "Critical"
    ):
        return "High"

    elif (
        row["Inventory_Status"] == "Critical"
        or
        row["Capacity_Status"] == "Critical"
    ):
        return "Medium"

    else:
        return "Low"


service_risk["Service_Risk"] = (
    service_risk.apply(
        classify_service_risk,
        axis=1
    )
)

In [ ]:
# Alert flag for customer fulfillment risk

service_risk["Service_Alert"] = np.where(
    service_risk["Service_Risk"] == "High",
    1,
    0
)

In [11]:
# Numerical representation of service risk severity

service_risk["Service_Risk_Score"] = (
    service_risk["Service_Risk"]
    .map(
        {
            "Low": 1,
            "Medium": 2,
            "High": 3
        }
    )
)

In [12]:
# Primary operational driver of service risk

def identify_risk_driver(row):

    if (
        row["Inventory_Status"] == "Critical"
        and
        row["Capacity_Status"] == "Critical"
    ):
        return "Inventory + Capacity"

    elif row["Inventory_Status"] == "Critical":
        return "Inventory"

    elif row["Capacity_Status"] == "Critical":
        return "Capacity"

    else:
        return "None"


service_risk["Risk_Driver"] = (
    service_risk.apply(
        identify_risk_driver,
        axis=1
    )
)

In [8]:
service_risk["Service_Risk"].value_counts()

Service_Risk
Low       4208
Medium    3304
High        96
Name: count, dtype: int64

In [9]:
service_risk["Service_Alert"].value_counts()

Service_Alert
0    7512
1      96
Name: count, dtype: int64

In [13]:
service_risk["Risk_Driver"].value_counts()

Risk_Driver
None                    4208
Inventory               3156
Capacity                 148
Inventory + Capacity      96
Name: count, dtype: int64

In [ ]:
#service_risk.to_csv(
#    "../data/processed/service_risk_weekly.csv",
#    index=False)